# Multi-Query Attention (多查询注意力) 实现

MQA是最极致的高效注意力机制，所有查询头共享同一对键值头。

**核心思想**：
- 所有查询头共享单一的KV头
- 最大化减少KV缓存（相比MHA减少num_heads倍）
- 推理速度最快，内存占用最小

**公式**：
$$
\text{MQA}(Q,K,V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h)W^O
$$
其中所有头使用同一个K和V。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
sns.set_style('whitegrid')

## 1. 辅助函数

In [ ]:
def softmax(x, axis=-1):
    """Softmax函数"""
    exp_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)


def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Scaled Dot-Product Attention
    
    Args:
        Q: Query矩阵 (seq_len, d_k)
        K: Key矩阵 (seq_len, d_k)
        V: Value矩阵 (seq_len, d_v)
        mask: 注意力掩码
    
    Returns:
        output: 输出 (seq_len, d_v)
        attention_weights: 注意力权重 (seq_len, seq_len)
    """
    d_k = Q.shape[-1]
    
    # 计算注意力得分
    scores = np.dot(Q, K.T) / np.sqrt(d_k)
    
    # 应用mask
    if mask is not None:
        scores = np.where(mask == 0, -1e9, scores)
    
    # Softmax
    attention_weights = softmax(scores, axis=-1)
    
    # 加权求和
    output = np.dot(attention_weights, V)
    
    return output, attention_weights

## 2. 实现Multi-Query Attention类

In [ ]:
class MultiQueryAttention:
    """
    Multi-Query Attention (多查询注意力)
    
    所有查询头共享单一的键值头
    """
    
    def __init__(self, embed_dim, num_heads, use_bias=True):
        """
        初始化MQA层
        
        Args:
            embed_dim: 嵌入维度
            num_heads: 查询头数量
            use_bias: 是否使用偏置
        """
        assert embed_dim % num_heads == 0, "embed_dim必须能被num_heads整除"
        
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.use_bias = use_bias
        
        # Q投影：完整的num_heads个头
        self.W_q = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        
        # K,V投影：只有1个头（MQA的核心）
        self.W_k = np.random.randn(embed_dim, self.head_dim) / np.sqrt(embed_dim)
        self.W_v = np.random.randn(embed_dim, self.head_dim) / np.sqrt(embed_dim)
        
        # 输出投影
        self.W_o = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        
        if use_bias:
            self.b_q = np.zeros(embed_dim)
            self.b_k = np.zeros(self.head_dim)
            self.b_v = np.zeros(self.head_dim)
            self.b_o = np.zeros(embed_dim)
    
    def split_heads(self, x):
        """分割查询成多个头"""
        seq_len = x.shape[0]
        x = x.reshape(seq_len, self.num_heads, self.head_dim)
        x = x.transpose(1, 0, 2)
        return x
    
    def combine_heads(self, x):
        """合并多个头"""
        x = x.transpose(1, 0, 2)
        seq_len = x.shape[0]
        x = x.reshape(seq_len, self.embed_dim)
        return x
    
    def forward(self, query, key=None, value=None, mask=None, return_attention=False):
        """
        前向传播
        
        Args:
            query: Query输入 (seq_len_q, embed_dim)
            key: Key输入
            value: Value输入
            mask: 注意力掩码
            return_attention: 是否返回注意力权重
        
        Returns:
            output: 输出
            attention_weights: (可选) 注意力权重
        """
        if key is None:
            key = query
        if value is None:
            value = key
        
        # 1. 线性投影
        Q = np.dot(query, self.W_q)  # (seq_len, embed_dim)
        K = np.dot(key, self.W_k)    # (seq_len, head_dim) - 只有1个头！
        V = np.dot(value, self.W_v)  # (seq_len, head_dim) - 只有1个头！
        
        if self.use_bias:
            Q += self.b_q
            K += self.b_k
            V += self.b_v
        
        # 2. 分割查询成多个头
        Q = self.split_heads(Q)  # (num_heads, seq_len, head_dim)
        # K和V保持单头，不分割
        
        # 3. MQA核心：所有查询头共享同一个KV
        head_outputs = []
        attention_weights_list = []
        
        for i in range(self.num_heads):
            Q_i = Q[i]
            
            # 关键：所有头使用相同的K和V
            head_output, attn_weights = scaled_dot_product_attention(
                Q_i, K, V, mask=mask
            )
            
            head_outputs.append(head_output)
            attention_weights_list.append(attn_weights)
        
        # 4. 合并头
        multi_head_output = np.stack(head_outputs, axis=0)
        concatenated = self.combine_heads(multi_head_output)
        
        # 5. 输出投影
        output = np.dot(concatenated, self.W_o)
        if self.use_bias:
            output += self.b_o
        
        if return_attention:
            attention_weights = np.stack(attention_weights_list, axis=0)
            return output, attention_weights
        
        return output
    
    def get_num_parameters(self):
        """计算参数量"""
        params = {
            'W_q': self.embed_dim * self.embed_dim,
            'W_k': self.embed_dim * self.head_dim,
            'W_v': self.embed_dim * self.head_dim,
            'W_o': self.embed_dim * self.embed_dim,
        }
        
        if self.use_bias:
            params['b_q'] = self.embed_dim
            params['b_k'] = self.head_dim
            params['b_v'] = self.head_dim
            params['b_o'] = self.embed_dim
        
        total = sum(params.values())
        return total, params


class MultiQuerySelfAttention(MultiQueryAttention):
    """MQA自注意力（简化接口）"""
    
    def forward(self, x, mask=None, return_attention=False):
        return super().forward(x, x, x, mask=mask, return_attention=return_attention)

## 3. 测试MQA

In [ ]:
# 参数设置
seq_len = 10
embed_dim = 64
num_heads = 8

print("配置:")
print(f"  序列长度: {seq_len}")
print(f"  嵌入维度: {embed_dim}")
print(f"  查询头数(Q): {num_heads}")
print(f"  键值头数(K,V): 1 ← MQA的核心！")
print(f"  每个头维度: {embed_dim // num_heads}")

# 生成输入
x = np.random.randn(seq_len, embed_dim)

# 创建MQA层
mqa = MultiQuerySelfAttention(embed_dim, num_heads)

# 前向传播
output, attn_weights = mqa.forward(x, return_attention=True)

print(f"\n形状:")
print(f"  输入: {x.shape}")
print(f"  输出: {output.shape}")
print(f"  注意力权重: {attn_weights.shape}")
print(f"    - {num_heads}个查询头")
print(f"    - 所有头共享1对KV头")

## 4. 可视化所有查询头的注意力模式

In [ ]:
# 可视化所有8个查询头的注意力
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i in range(8):
    sns.heatmap(attn_weights[i], annot=True, fmt='.2f', cmap='YlOrRd', 
                ax=axes[i], cbar=True, square=True, vmin=0, vmax=0.5)
    axes[i].set_title(f'Q头{i+1} (共享KV头)', fontweight='bold')
    axes[i].set_xlabel('Key位置')
    axes[i].set_ylabel('Query位置')

plt.suptitle('MQA: 所有查询头共享同一KV头', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("观察:")
print("  ✓ 所有查询头都使用相同的K和V")
print("  ✓ 但通过不同的Q投影，仍能学到不同的注意力模式")
print("  ✓ 这是MQA能够工作的关键原因")

## 5. 对比MHA、GQA、MQA

In [ ]:
# 创建三种变体进行对比
# 注意：这里用不同的类来模拟MHA和GQA

# MHA: 所有头独立 (模拟)
mha_params = 4 * embed_dim * embed_dim
mha_kv_cache = 2 * seq_len * num_heads * (embed_dim // num_heads)

# GQA: 2个KV头 (模拟)
num_kv_heads_gqa = 2
gqa_params = 2 * embed_dim * embed_dim + 2 * embed_dim * num_kv_heads_gqa * (embed_dim // num_heads)
gqa_kv_cache = 2 * seq_len * num_kv_heads_gqa * (embed_dim // num_heads)

# MQA: 1个KV头
mqa_params, _ = mqa.get_num_parameters()
mqa_kv_cache = 2 * seq_len * 1 * (embed_dim // num_heads)

# 对比表
print("注意力变体对比:\n")
print(f"{'变体':<15} {'Q头':>5} {'KV头':>5} {'参数量':>12} {'KV缓存':>12} {'缓存节省':>10}")
print("-" * 65)

variants_data = [
    ("MHA", num_heads, num_heads, mha_params, mha_kv_cache, 0),
    ("GQA", num_heads, num_kv_heads_gqa, gqa_params, gqa_kv_cache, 
     (1 - gqa_kv_cache/mha_kv_cache)*100),
    ("MQA", num_heads, 1, mqa_params, mqa_kv_cache, 
     (1 - mqa_kv_cache/mha_kv_cache)*100),
]

for name, q_heads, kv_heads, params, kv_cache, saving in variants_data:
    print(f"{name:<15} {q_heads:>5} {kv_heads:>5} {params:>12,} {kv_cache:>12,} {saving:>9.1f}%")

print("\n关键发现:")
print(f"  ✓ MQA节省KV缓存: {(1 - mqa_kv_cache/mha_kv_cache)*100:.1f}%")
print(f"  ✓ MQA参数量减少: {(1 - mqa_params/mha_params)*100:.1f}%")
print(f"  ✓ MQA是最极致的优化方案")

## 6. 可视化参数量和KV缓存对比

In [ ]:
# 绘制对比图
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

names = ["MHA\n(8 KV)", "GQA\n(2 KV)", "MQA\n(1 KV)"]
params_list = [mha_params, gqa_params, mqa_params]
kv_caches = [mha_kv_cache, gqa_kv_cache, mqa_kv_cache]
colors = ['#ff7f0e', '#1f77b4', '#d62728']

# 参数量对比
bars1 = ax1.bar(names, params_list, color=colors, alpha=0.7, edgecolor='black')
ax1.set_ylabel('参数量', fontsize=12)
ax1.set_title('参数量对比', fontsize=14, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

for bar in bars1:
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
             f'{int(height):,}',
             ha='center', va='bottom', fontsize=10)

# KV缓存对比
bars2 = ax2.bar(names, kv_caches, color=colors, alpha=0.7, edgecolor='black')
ax2.set_ylabel('KV缓存大小', fontsize=12)
ax2.set_title(f'KV缓存对比 (seq_len={seq_len})', fontsize=14, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

for bar in bars2:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
             f'{int(height):,}',
             ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

print("可视化说明:")
print("  左图：MQA显著减少参数量")
print("  右图：MQA的KV缓存是MHA的1/8（推理速度的关键）")

## 7. 长序列下的KV缓存分析

In [ ]:
# 不同序列长度下的KV缓存
seq_lengths = [128, 512, 1024, 2048, 4096]
head_dim = embed_dim // num_heads

plt.figure(figsize=(12, 6))

# MHA
mha_caches = [2 * s * num_heads * head_dim * 4 / 1024 / 1024 for s in seq_lengths]
plt.plot(seq_lengths, mha_caches, marker='o', linewidth=2, 
         markersize=8, label='MHA (8 KV头)', color='#ff7f0e')

# GQA (2 KV heads)
gqa_caches = [2 * s * 2 * head_dim * 4 / 1024 / 1024 for s in seq_lengths]
plt.plot(seq_lengths, gqa_caches, marker='s', linewidth=2, 
         markersize=8, label='GQA (2 KV头)', color='#1f77b4')

# MQA
mqa_caches = [2 * s * 1 * head_dim * 4 / 1024 / 1024 for s in seq_lengths]
plt.plot(seq_lengths, mqa_caches, marker='^', linewidth=2, 
         markersize=8, label='MQA (1 KV头)', color='#d62728')

plt.xlabel('序列长度', fontsize=12)
plt.ylabel('KV缓存大小 (MB)', fontsize=12)
plt.title('不同序列长度下的KV缓存对比', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("长序列分析:")
print(f"  在序列长度4096时:")
print(f"    MHA: {mha_caches[-1]:.2f} MB")
print(f"    GQA (2头): {gqa_caches[-1]:.2f} MB")
print(f"    MQA: {mqa_caches[-1]:.2f} MB")
print(f"\n  MQA节省: {mha_caches[-1] - mqa_caches[-1]:.2f} MB ({(1-mqa_caches[-1]/mha_caches[-1])*100:.1f}%)")
print("\n  ✓ 序列越长，MQA的优势越明显")
print("  ✓ 对于超长文本生成至关重要")

## 8. 共享KV的影响分析

In [ ]:
# 分析不同查询头的注意力模式
query_pos = 5

plt.figure(figsize=(14, 6))

for i in range(num_heads):
    attention_pattern = attn_weights[i, query_pos, :]
    plt.plot(attention_pattern, marker='o', label=f'Q头{i+1}', alpha=0.7, linewidth=2)

plt.xlabel('Key位置', fontsize=12)
plt.ylabel('注意力权重', fontsize=12)
plt.title(f'MQA: Query位置{query_pos}在不同头中的注意力分布', fontsize=14, fontweight='bold')
plt.legend(ncol=4, loc='upper right', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nQuery位置{query_pos}的分析:")
for i in range(num_heads):
    most_attended = np.argmax(attn_weights[i, query_pos, :])
    print(f"  Q头{i+1}: 最关注位置{most_attended} "
          f"(权重={attn_weights[i, query_pos, most_attended]:.3f})")

print("\n关键观察:")
print("  ✓ 虽然所有头共享KV，但不同Q投影产生不同的注意力模式")
print("  ✓ 这是MQA能够保持一定表达能力的原因")
print("  ✓ 但相比MHA/GQA，多样性仍然受限")

## 9. 因果Mask下的MQA

In [ ]:
def create_causal_mask(seq_len):
    """创建因果mask"""
    return np.tril(np.ones((seq_len, seq_len)))

# 应用因果mask
causal_mask = create_causal_mask(seq_len)
output_masked, attn_masked = mqa.forward(x, mask=causal_mask, return_attention=True)

# 可视化
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 因果mask
sns.heatmap(causal_mask, annot=False, cmap='Greys', 
            ax=axes[0], cbar=False, square=True)
axes[0].set_title('因果Mask', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Key位置')
axes[0].set_ylabel('Query位置')

# 无mask
sns.heatmap(attn_weights[0], annot=True, fmt='.2f', cmap='YlOrRd', 
            ax=axes[1], square=True)
axes[1].set_title('Q头1 (无Mask)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Key位置')
axes[1].set_ylabel('Query位置')

# 有mask
sns.heatmap(attn_masked[0], annot=True, fmt='.2f', cmap='YlOrRd', 
            ax=axes[2], square=True)
axes[2].set_title('Q头1 (带因果Mask)', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Key位置')
axes[2].set_ylabel('Query位置')

plt.tight_layout()
plt.show()

print("因果Mask + MQA:")
print("  ✓ 自回归生成的标准配置")
print("  ✓ MQA显著减少KV缓存，加速生成")
print("  ✓ 用于PaLM、Falcon等模型")

## 10. 实际应用：PaLM配置

In [ ]:
# PaLM 540B使用MQA
palm_embed_dim = 8192
palm_num_heads = 32
palm_seq_len = 2048

print("PaLM 540B配置:\n")
print(f"  嵌入维度: {palm_embed_dim}")
print(f"  查询头数: {palm_num_heads}")
print(f"  键值头数: 1 (MQA)")
print(f"  序列长度: {palm_seq_len}")

# 计算参数和缓存
palm_head_dim = palm_embed_dim // palm_num_heads
palm_mqa_params = 2 * palm_embed_dim ** 2 + 2 * palm_embed_dim * palm_head_dim

print(f"\n  单层MQA参数量: {palm_mqa_params:,}")
print(f"  118层总参数量: {palm_mqa_params * 118:,}")

# KV缓存对比
palm_mha_cache = 2 * palm_seq_len * palm_num_heads * palm_head_dim * 4 / 1024 / 1024
palm_mqa_cache = 2 * palm_seq_len * 1 * palm_head_dim * 4 / 1024 / 1024

print(f"\nKV缓存对比 (序列长度={palm_seq_len}, FP32):")
print(f"  如果使用MHA: {palm_mha_cache:.1f} MB")
print(f"  实际MQA缓存: {palm_mqa_cache:.1f} MB")
print(f"  节省: {palm_mha_cache - palm_mqa_cache:.1f} MB ({(1 - palm_mqa_cache/palm_mha_cache)*100:.1f}%)")

print("\nPaLM为何选择MQA:")
print("  ✓ 540B超大规模模型，KV缓存是瓶颈")
print("  ✓ MQA最大化减少内存占用")
print("  ✓ 显著提升推理速度")
print("  ✓ 实验表明质量损失可接受")

## 11. MQA的优缺点总结

In [ ]:
# 创建对比表
import pandas as pd

data = {
    '特性': ['Q头数', 'KV头数', 'KV缓存', '参数量', '推理速度', '模型质量', '表达能力'],
    'MHA': ['h', 'h', '100%', '最多', '慢', '最好', '最强'],
    'GQA': ['h', 'h/G', f'{100/4:.0f}%', '中等', '快', '接近MHA', '强'],
    'MQA': ['h', '1', f'{100/8:.1f}%', '最少', '最快', '可能下降', '受限']
}

df = pd.DataFrame(data)
print("\nMHA vs GQA vs MQA 综合对比:\n")
print(df.to_string(index=False))

print("\n\nMQA的优势:")
print("  ✅ KV缓存最小（相比MHA减少num_heads倍）")
print("  ✅ 推理速度最快")
print("  ✅ 参数量最少")
print("  ✅ 内存占用最小")
print("  ✅ 适合超大规模模型")

print("\nMQA的劣势:")
print("  ⚠️ 可能轻微降低模型质量")
print("  ⚠️ 所有Q头共享KV限制表达能力")
print("  ⚠️ 某些任务上不如MHA/GQA")

print("\n最佳实践:")
print("  • 超大模型（>100B参数）→ 考虑MQA")
print("  • 长序列生成 → 考虑MQA")
print("  • 资源极度受限 → 选择MQA")
print("  • 质量和效率平衡 → 选择GQA")
print("  • 质量优先 → 选择MHA")

## 总结

### MQA的核心特点：

1. **极致效率**：
   - 所有Q头共享1对KV头
   - KV缓存减少num_heads倍
   - 推理速度最快

2. **参数优化**：
   - K,V投影只有1个头的维度
   - 显著减少参数量

3. **质量权衡**：
   - 通过不同的Q投影保持多样性
   - 可能轻微降低质量
   - 实践中效果可接受

### 关键公式：
$$
\text{KV缓存} = 2 \times \text{seq\_len} \times 1 \times \text{head\_dim}
$$

### 应用场景：
- PaLM、Falcon、StarCoder等超大模型
- 需要极致推理速度的场景
- 资源受限的部署环境
- 超长序列生成任务